# Satellite ROI Maps for Detroit Lake and Upper Klamath Lake

This notebook contains six maps showing the regions of interest (ROIs) for satellite data extraction:

### Detroit Lake:
1. Sentinel-2: 500 m × 500 m area centered at (-122.184°, 44.711°)
2. Landsat: 500 m × 500 m area centered at (-122.184°, 44.711°)
3. MODIS: Single 500 m × 500 m pixel centered at (-122.184°, 44.711°)

### Upper Klamath Lake:
4. Sentinel-2: 500 m × 500 m area centered at (-121.900°, 42.400°)
5. Landsat: 500 m × 500 m area centered at (-121.900°, 42.400°)
6. MODIS: Single 500 m × 500 m pixel centered at (-121.900°, 42.400°)

**Note**: For Sentinel-2 and Landsat, the ROI represents an area where all pixels within the 500 m × 500 m box are averaged. For MODIS, the ROI represents a single 500 m pixel at the specified location.

In [1]:
import ee
import folium
import math

# Initialize Google Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')

In [2]:
# Define lake locations (from original notebook)
lakes = [
    {
        'name': 'Detroit Lake',
        'lon': -122.184,
        'lat': 44.711
    },
    {
        'name': 'Upper Klamath Lake',
        'lon': -121.900,
        'lat': 42.400
    }
]

# Buffer size for 500x500m box (250m radius)
buffer_size = 250  # meters

## Helper Functions for ROI Creation

In [3]:
def add_ee_layer(self, ee_image_object, vis_params, name):
    """Adds Earth Engine layers to folium map."""
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
        name=name,
        overlay=True,
        control=True
    ).add_to(self)

# Add the method to folium Map class
folium.Map.add_ee_layer = add_ee_layer

def create_500m_box(lon, lat):
    """Create a 500 m x 500 m box geometry centered at given coordinates."""
    point = ee.Geometry.Point([lon, lat])
    box = point.buffer(buffer_size).bounds()
    return box

def create_modis_pixel(lon, lat):
    """Create a single MODIS pixel (500 m x 500 m) aligned to MODIS grid."""
    # MODIS pixels are approximately 500m x 500m
    # We'll create a square that represents one MODIS pixel
    point = ee.Geometry.Point([lon, lat])
    
    # Create a 500m x 500m square centered at the point
    # Note: MODIS pixels are actually aligned to a specific grid,
    # but for visualization we'll show a 500m square at the location
    pixel = point.buffer(250, 1).bounds()
    return pixel

## Sentinel-2 ROI Maps (500 m × 500 m area)

In [10]:
# Sentinel-2 Detroit Lake
detroit = lakes[0]
m_s2_detroit = folium.Map(location=[detroit['lat'], detroit['lon']], zoom_start=14)

# Create 500x500m ROI
roi_detroit = create_500m_box(detroit['lon'], detroit['lat'])

# Convert to GeoJSON for folium
roi_geojson = roi_detroit.getInfo()

# Add ROI to map
folium.GeoJson(
    roi_geojson,
    name='500 m × 500 m ROI',
    style_function=lambda x: {
        'fillColor': 'gold',
        'color': 'gold',
        'weight': 2,
        'fillOpacity': 0.3
    }
).add_to(m_s2_detroit)

# Add center marker
folium.Marker(
    [detroit['lat'], detroit['lon']],
    popup=f"Detroit Lake Center<br>Lon: {detroit['lon']}<br>Lat: {detroit['lat']}",
    icon=folium.Icon(color='red', icon='crosshairs', prefix='fa')
).add_to(m_s2_detroit)

# Add title
title_html = '''<h4 style="position: fixed; 
                top: 10px; left: 50px; width: 300px; 
                background-color: white; z-index: 1000; 
                padding: 10px; border-radius: 5px;
                box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
                Sentinel-2: Detroit Lake ROI</h4>'''
m_s2_detroit.get_root().html.add_child(folium.Element(title_html))

print("Sentinel-2 Detroit Lake ROI Map")
m_s2_detroit
m_s2_detroit.save('sentinel2_detroit_map.html')

Sentinel-2 Detroit Lake ROI Map


In [5]:
# Sentinel-2 Upper Klamath Lake
klamath = lakes[1]
m_s2_klamath = folium.Map(location=[klamath['lat'], klamath['lon']], zoom_start=14)

# Create 500x500m ROI
roi_klamath = create_500m_box(klamath['lon'], klamath['lat'])
roi_geojson = roi_klamath.getInfo()

# Add ROI to map
folium.GeoJson(
    roi_geojson,
    name='500 m × 500 m ROI',
    style_function=lambda x: {
        'fillColor': 'gold',
        'color': 'gold',
        'weight': 2,
        'fillOpacity': 0.3
    }
).add_to(m_s2_klamath)

# Add center marker
folium.Marker(
    [klamath['lat'], klamath['lon']],
    popup=f"Upper Klamath Lake Center<br>Lon: {klamath['lon']}<br>Lat: {klamath['lat']}",
    icon=folium.Icon(color='red', icon='crosshairs', prefix='fa')
).add_to(m_s2_klamath)

# Add title
title_html = '''<h4 style="position: fixed; 
                top: 10px; left: 50px; width: 350px; 
                background-color: white; z-index: 1000; 
                padding: 10px; border-radius: 5px;
                box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
                Sentinel-2: Upper Klamath Lake ROI</h4>'''
m_s2_klamath.get_root().html.add_child(folium.Element(title_html))

print("Sentinel-2 Upper Klamath Lake ROI Map")
m_s2_klamath

Sentinel-2 Upper Klamath Lake ROI Map


## Landsat ROI Maps (500 m × 500 m area)

In [6]:
# Landsat Detroit Lake
m_ls_detroit = folium.Map(location=[detroit['lat'], detroit['lon']], zoom_start=14)

# Create 500x500m ROI (same as Sentinel-2)
roi_detroit = create_500m_box(detroit['lon'], detroit['lat'])
roi_geojson = roi_detroit.getInfo()

# Add ROI to map
folium.GeoJson(
    roi_geojson,
    name='500 m × 500 m ROI',
    style_function=lambda x: {
        'fillColor': 'gold',
        'color': 'gold',
        'weight': 2,
        'fillOpacity': 0.3
    }
).add_to(m_ls_detroit)

# Add center marker
folium.Marker(
    [detroit['lat'], detroit['lon']],
    popup=f"Detroit Lake Center<br>Lon: {detroit['lon']}<br>Lat: {detroit['lat']}",
    icon=folium.Icon(color='blue', icon='crosshairs', prefix='fa')
).add_to(m_ls_detroit)

# Add title
title_html = '''<h4 style="position: fixed; 
                top: 10px; left: 50px; width: 300px; 
                background-color: white; z-index: 1000; 
                padding: 10px; border-radius: 5px;
                box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
                Landsat: Detroit Lake ROI</h4>'''
m_ls_detroit.get_root().html.add_child(folium.Element(title_html))

print("Landsat Detroit Lake ROI Map")
m_ls_detroit

Landsat Detroit Lake ROI Map


In [7]:
# Landsat Upper Klamath Lake
m_ls_klamath = folium.Map(location=[klamath['lat'], klamath['lon']], zoom_start=14)

# Create 500x500m ROI
roi_klamath = create_500m_box(klamath['lon'], klamath['lat'])
roi_geojson = roi_klamath.getInfo()

# Add ROI to map
folium.GeoJson(
    roi_geojson,
    name='500 m × 500 m ROI',
    style_function=lambda x: {
        'fillColor': 'gold',
        'color': 'gold',
        'weight': 2,
        'fillOpacity': 0.3
    }
).add_to(m_ls_klamath)

# Add center marker
folium.Marker(
    [klamath['lat'], klamath['lon']],
    popup=f"Upper Klamath Lake Center<br>Lon: {klamath['lon']}<br>Lat: {klamath['lat']}",
    icon=folium.Icon(color='blue', icon='crosshairs', prefix='fa')
).add_to(m_ls_klamath)

# Add title
title_html = '''<h4 style="position: fixed; 
                top: 10px; left: 50px; width: 350px; 
                background-color: white; z-index: 1000; 
                padding: 10px; border-radius: 5px;
                box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
                Landsat: Upper Klamath Lake ROI</h4>'''
m_ls_klamath.get_root().html.add_child(folium.Element(title_html))

print("Landsat Upper Klamath Lake ROI Map")
m_ls_klamath

Landsat Upper Klamath Lake ROI Map


## MODIS ROI Maps (Single 500 m x 500 m pixel)

In [8]:
# MODIS Detroit Lake
m_modis_detroit = folium.Map(location=[detroit['lat'], detroit['lon']], zoom_start=14)

# Create single MODIS pixel (500x500m)
roi_detroit = create_modis_pixel(detroit['lon'], detroit['lat'])
roi_geojson = roi_detroit.getInfo()

# Add ROI to map
folium.GeoJson(
    roi_geojson,
    name='MODIS 500 m x 500 m Pixel',
    style_function=lambda x: {
        'fillColor': 'gold',
        'color': 'gold',
        'weight': 2,
        'fillOpacity': 0.3
    }
).add_to(m_modis_detroit)

# Add center marker
folium.Marker(
    [detroit['lat'], detroit['lon']],
    popup=f"Detroit Lake Center<br>Lon: {detroit['lon']}<br>Lat: {detroit['lat']}<br>(MODIS single pixel)",
    icon=folium.Icon(color='green', icon='crosshairs', prefix='fa')
).add_to(m_modis_detroit)

# Add title
title_html = '''<h4 style="position: fixed; 
                top: 10px; left: 50px; width: 300px; 
                background-color: white; z-index: 1000; 
                padding: 10px; border-radius: 5px;
                box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
                MODIS: Detroit Lake ROI<br>
                <small>(Single 500m pixel)</small></h4>'''
m_modis_detroit.get_root().html.add_child(folium.Element(title_html))

print("MODIS Detroit Lake ROI Map (Single Pixel)")
m_modis_detroit

MODIS Detroit Lake ROI Map (Single Pixel)


In [9]:
# MODIS Upper Klamath Lake
m_modis_klamath = folium.Map(location=[klamath['lat'], klamath['lon']], zoom_start=14)

# Create single MODIS pixel (500x500m)
roi_klamath = create_modis_pixel(klamath['lon'], klamath['lat'])
roi_geojson = roi_klamath.getInfo()

# Add ROI to map
folium.GeoJson(
    roi_geojson,
    name='MODIS 500 m x 500 m Pixel',
    style_function=lambda x: {
        'fillColor': 'gold',
        'color': 'gold',
        'weight': 2,
        'fillOpacity': 0.3
    }
).add_to(m_modis_klamath)

# Add center marker
folium.Marker(
    [klamath['lat'], klamath['lon']],
    popup=f"Upper Klamath Lake Center<br>Lon: {klamath['lon']}<br>Lat: {klamath['lat']}<br>(MODIS single pixel)",
    icon=folium.Icon(color='green', icon='crosshairs', prefix='fa')
).add_to(m_modis_klamath)

# Add title
title_html = '''<h4 style="position: fixed; 
                top: 10px; left: 50px; width: 350px; 
                background-color: white; z-index: 1000; 
                padding: 10px; border-radius: 5px;
                box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
                MODIS: Upper Klamath Lake ROI<br>
                <small>(Single 500m pixel)</small></h4>'''
m_modis_klamath.get_root().html.add_child(folium.Element(title_html))

print("MODIS Upper Klamath Lake ROI Map (Single Pixel)")
m_modis_klamath

MODIS Upper Klamath Lake ROI Map (Single Pixel)
